In [1]:
from datasets import load_dataset_builder, load_dataset, get_dataset_split_names
from log_wrapper import log_calls
from huggingface_hub import hf_hub_download
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
)

import pandas as pd

/Users/vladpalamarchuk/anaconda3/envs/dev/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
ds = load_dataset("yandex/yambda", "flat-multievent-50m", split="train")
df = ds.to_pandas()

artist_map = pd.read_parquet("hf://datasets/yandex/yambda/artist_item_mapping.parquet")
album_map = pd.read_parquet("hf://datasets/yandex/yambda/album_item_mapping.parquet")

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47790449 entries, 0 to 47790448
Data columns (total 7 columns):
 #   Column                Dtype  
---  ------                -----  
 0   uid                   uint32 
 1   timestamp             uint32 
 2   item_id               uint32 
 3   is_organic            uint8  
 4   played_ratio_pct      float64
 5   track_length_seconds  float64
 6   event_type            object 
dtypes: float64(2), object(1), uint32(3), uint8(1)
memory usage: 1.6+ GB


### Колонки в датасете:
1. **uid** - анонимизированный идентификатор пользователя
2. **timestamp** - временная метка события
3. **item_id** - идентификатор музыкального трека
4. **is_organic** - флаг органического взаимодействия (не из рекомендаций)
5. **played_ratio_pct** - процент прослушанного трека (0-100)
6. **track_length_seconds** - длительность трека в секундах
7. **event_type** - тип события:
   - прослушивания (listen)
   - лайки (like)
   - дизлайки (dislike)
   - отмены лайков (unlike)
   - отмены дизлайков (undislike)

In [21]:
df.sample(5)

,uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds,event_type
16510210,345100,12055940,6332689,0,25.0,160.0,listen
11284285,236600,8613030,8190641,1,0.0,215.0,listen
21056232,436200,22911230,8922494,1,100.0,30.0,listen
17382988,361600,8931085,3203533,1,5.0,185.0,listen
45301227,945800,25995570,2718138,1,100.0,1990.0,listen


In [6]:
print(f"""
Распределение типов событий:
{df['event_type'].value_counts(normalize=True)}

Уникальных пользователей: {df['uid'].nunique():,}
Уникальных треков: {df['item_id'].nunique():,}

Доля органических взаимодействий: {df['is_organic'].mean():.2%}
""")


Распределение типов событий:
event_type
listen       0.972312
like         0.018444
unlike       0.006549
dislike      0.002255
undislike    0.000440
Name: proportion, dtype: float64

Уникальных пользователей: 10,000
Уникальных треков: 934,057

Доля органических взаимодействий: 52.05%



**Пробуем предсказывать лайки просто через самые популярные треки**

In [ ]:
df.groupby('A')['B'] 

**Catboost**

In [ ]:
# Бинарный таргет: был ли лайк на этот трек от этого юзера
liked_pairs = df[df["event_type"] == "like"][["uid", "item_id"]].drop_duplicates()
liked_pairs['liked'] = 1

# Берем только прослушивания
listens = df[df["event_type"] == "listen"].copy()

# Мерджим — если был лайк на эту пару (uid, item_id), то liked=1
listens = listens.merge(liked_pairs, on=["uid", "item_id"], how="left")
listens["liked"] = listens["liked"].fillna(0).astype(int)


In [ ]:
# Сортируем по времени 
listens = listens.sort_values("timestamp")

split_index = int(len(listens) * 0.8)

X = listens.drop(columns=["event_type", "liked"])
y = listens["liked"]

X_train, y_train = X.iloc[:split_index], y.iloc[:split_index]
X_test, y_test = X.iloc[split_index:], y.iloc[split_index:]


In [16]:
model = CatBoostClassifier(
    n_estimators=100,
    random_state=42,
    cat_features=["is_organic"],
    auto_class_weights='Balanced'
)
model.fit(X_train, y_train)

Learning rate set to 0.5
0:	learn: 0.6168449	total: 1.04s	remaining: 1m 43s
1:	learn: 0.5947466	total: 1.67s	remaining: 1m 21s
2:	learn: 0.5875706	total: 2.16s	remaining: 1m 9s
3:	learn: 0.5849899	total: 2.52s	remaining: 1m
4:	learn: 0.5829390	total: 2.93s	remaining: 55.7s
5:	learn: 0.5821760	total: 3.35s	remaining: 52.5s
6:	learn: 0.5816521	total: 3.72s	remaining: 49.4s
7:	learn: 0.5810556	total: 4.05s	remaining: 46.5s
8:	learn: 0.5805207	total: 4.43s	remaining: 44.8s
9:	learn: 0.5799196	total: 4.93s	remaining: 44.4s
10:	learn: 0.5797314	total: 5.41s	remaining: 43.8s
11:	learn: 0.5793616	total: 5.77s	remaining: 42.3s
12:	learn: 0.5791071	total: 6.19s	remaining: 41.4s
13:	learn: 0.5788590	total: 6.52s	remaining: 40.1s
14:	learn: 0.5785652	total: 6.9s	remaining: 39.1s
15:	learn: 0.5783366	total: 7.25s	remaining: 38s
16:	learn: 0.5781512	total: 7.63s	remaining: 37.2s
17:	learn: 0.5780310	total: 7.95s	remaining: 36.2s
18:	learn: 0.5778533	total: 8.38s	remaining: 35.7s
19:	learn: 0.5776613

In [ ]:
# Вероятности нужны для AUC метрик
preds_proba = model.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, preds_proba))
print("PR-AUC:", average_precision_score(y_test, preds_proba))
print(classification_report(y_test, model.predict(X_test)))

ROC-AUC: 0.7783671960132067
PR-AUC: 0.4779389142525744
              precision    recall  f1-score   support

           0       0.93      0.61      0.74   7004317
           1       0.42      0.87      0.57   2289126

    accuracy                           0.67   9293443
   macro avg       0.68      0.74      0.65   9293443
weighted avg       0.81      0.67      0.69   9293443

